In [1]:
import numpy as np
import pandas as pd
import re
import math
#country = input("Please write down the name of the country you want to know about")
# read csv file
Energy_resources_data = pd.read_csv("各國家資訊.csv", encoding="utf-8")
# display(Energy_resources_data)

def clean_hours(hours_str):
    numbers = re.findall(r'\d+', hours_str.replace(',', ''))
    if not numbers:
        return 0
    nums = [int(num) for num in numbers]
    return sum(nums)/len(nums)

def get_solar_correction(latitude,day_of_year):
    lat_rad = math.radians(latitude)
    # 計算赤緯角：隨日期變化的太陽位置
    declination = 23.45 * math.sin(math.radians((360/365)*(284+day_of_year)))
    decl_rad = math.radians(declination)
    correction_factor = math.cos(lat_rad - decl_rad)
    #確保極區不會出現負值
    return max(0, correction_factor)

if "df" in locals():
    Energy_resources_data['日照時數_數值']=Energy_resources_data['年均日照時常'].apply(clean_hours)

# 假設：1平方米面板 (A=1), 18% 轉換效率 (r=0.18), 75% 系統表現係數 (PR=0.75)
target_day_of_year = 172
panel_area = 1
efficiency = 0.18
pr = 0.75
Energy_resources_data['角度修正係數'] = Energy_resources_data['緯度 (Latitude)'].apply(lambda x: get_solar_correction(x, target_day_of_year))
Energy_resources_data['修正後日發電預估_khw'] = (Energy_resources_data['日照時間_平均']/365) * panel_area * efficiency * pr * Energy_resources_data['角度修正係數']
# Energy_resources_data['年預計發電量_kWh'] = Energy_resources_data['修正後日發電預估_khw'] * 365

df_result = Energy_resources_data.sort_values(by='修正後日發電預估_khw', ascending=False)[['國家', '修正後日發電預估_khw']]
print("--- 各國家太陽能發電潛力排名 ---")
print(df_result[['國家名稱', '緯度 (Latitude)', '角度修正係數', '修正後日發電預估_khw']])



KeyError: '日照時間_平均'